# Complete Fine-tuning Pipeline

This notebook runs the complete workflow from dataset creation to model export.

**Features:**
- Auto-saves all intermediate data
- Can resume from any checkpoint by loading saved data
- Flexible workflow: run all steps or skip to specific stages

**Data Flow:**
- Augmented data → `data/generated/augmented_dataset.json`
- Filtered data → `data/generated/filtered_dataset.json`
- Converted data → `data/processed/converted_dataset.json`
- Train/Val splits → `data/processed/train.json` and `data/processed/val.json`

In [1]:
import sys
import os
from pathlib import Path
import json

# Add project root to path
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Added to path: {project_root / 'src'}")

from datasets import Dataset
_original_map = Dataset.map
def _patched_map(self, *args, **kwargs):
    kwargs['num_proc'] = None
    return _original_map(self, *args, **kwargs)
Dataset.map = _patched_map

from src.utils import setup_logging, load_config, ensure_dir, get_project_root
from src.dataset_creation import load_initial_dataset, augment_dataset, filter_quality, save_dataset
from src.dataset_preparation import convert_to_unsloth_format, split_dataset, validate_dataset, save_processed_dataset
from src.training import load_model, setup_lora, train_model, save_checkpoint
from src.export import export_all

setup_logging()

Project root: d:\LFM-Tuner
Added to path: d:\LFM-Tuner\src
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0126 12:40:00.256000 13220 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# Load configuration
config = load_config()
print("Configuration loaded:")
print(f"  Model: {config['model']['name']}")
print(f"  Attention Backend: {config['attention_backend']}")
print(f"  Quantization: 16bit={config['quantization']['load_in_16bit']}")

Configuration loaded:
  Model: LiquidAI/LFM2.5-1.2B-Thinking
  Attention Backend: sageattention
  Quantization: 16bit=True


## Step 1: Load Initial Dataset

Start here with your raw dataset.

In [ ]:
# Resolve dataset path
resolved_path = project_root / config['dataset']['raw_path'] / 'initial_dataset.json'
print(f"Loading dataset from: {resolved_path}")

if not resolved_path.exists():
    raise FileNotFoundError(f"Dataset file not found at: {resolved_path}")

# Load initial dataset
initial_data = load_initial_dataset(resolved_path)
print(f"✓ Loaded {len(initial_data)} initial examples")

## Step 2: Augmentation (Optional)

Choose one:
- **Option A**: Run augmentation and continue with live data
- **Option B**: Load previously augmented data
- **Option C**: Skip augmentation entirely

In [ ]:
# OPTION A: Run augmentation (uncomment to use)
#
# Augmentation Strategies:
# - 'paraphrase': Rewrites instructions/prompts with different wording (same meaning)
#                 Good for: Increasing diversity without changing task
#                 Works with: instruction, completion schemas
#
# - 'expand': Generates new detailed responses for existing instructions
#            Good for: Creating more comprehensive training examples
#            Works with: instruction, completion schemas
#
# - 'variation': Creates new instructions asking similar things differently
#               Good for: Teaching model to handle different question phrasings
#               Works with: instruction schema only
#
# - 'response_variation': Generates alternative responses with different styles
#                        Good for: Teaching multiple valid response approaches
#                        Works with: instruction schema only

augmented_data = augment_dataset(
    initial_data,
    api_url=config['lm_studio']['api_url'],
    augmentation_strategy='variation',  # Choose based on your needs (see above)
    num_augmentations_per_example=100,
    save_path=project_root / 'data/generated/augmented_dataset.json',
    save_format='json'
)
print(f"✓ Augmented and saved: {len(augmented_data)} examples")

In [ ]:
# OPTION B: Load previously augmented data (uncomment to use)
# augmented_path = project_root / 'data/generated/augmented_dataset.json'
# if augmented_path.exists():
#     with open(augmented_path, 'r', encoding='utf-8') as f:
#         augmented_data = json.load(f)
#     print(f"✓ Loaded augmented data: {len(augmented_data)} examples")
# else:
#     print(f"⚠️ Augmented data not found at {augmented_path}")
#     augmented_data = initial_data  # Fallback to initial data

In [ ]:
# OPTION C: Skip augmentation (use this if you don't want to augment)
# augmented_data = initial_data
# print(f"✓ Using initial data without augmentation: {len(augmented_data)} examples")

## Step 3: Quality Filtering

Choose one:
- **Option A**: Run filtering and continue with live data
- **Option B**: Load previously filtered data

In [ ]:
# OPTION A: Run filtering (default)
filtered_data = filter_quality(
    augmented_data,
    min_length=10,
    max_length=2000,
    remove_duplicates=True,
    save_path=project_root / 'data/generated/filtered_dataset.json',
    save_format='json'
)
print(f"✓ Filtered and saved: {len(filtered_data)} examples")

In [ ]:
# OPTION B: Load previously filtered data (uncomment to use)
# filtered_path = project_root / 'data/generated/filtered_dataset.json'
# if filtered_path.exists():
#     with open(filtered_path, 'r', encoding='utf-8') as f:
#         filtered_data = json.load(f)
#     print(f"✓ Loaded filtered data: {len(filtered_data)} examples")
# else:
#     print(f"⚠️ Filtered data not found at {filtered_path}")
#     filtered_data = augmented_data  # Fallback

## Step 4: Format Conversion

Choose one:
- **Option A**: Run conversion and continue with live data
- **Option B**: Load previously converted data

In [ ]:
# OPTION A: Run conversion (default)
converted_data = convert_to_unsloth_format(
    filtered_data,
    model_type=config['dataset']['model_type'],
    format_type=config['dataset']['format_type'],
    save_path=project_root / 'data/processed/converted_dataset.json',
    save_format='json'
)
print(f"✓ Converted and saved: {len(converted_data)} examples")

In [ ]:
# OPTION B: Load previously converted data (uncomment to use)
# converted_path = project_root / 'data/processed/converted_dataset.json'
# if converted_path.exists():
#     with open(converted_path, 'r', encoding='utf-8') as f:
#         converted_data = json.load(f)
#     print(f"✓ Loaded converted data: {len(converted_data)} examples")
# else:
#     print(f"⚠️ Converted data not found at {converted_path}")
#     # Need to convert
#     converted_data = convert_to_unsloth_format(
#         filtered_data,
#         model_type=config['dataset']['model_type'],
#         format_type=config['dataset']['format_type'],
#         save_path=converted_path,
#         save_format='json'
#     )
#     print(f"✓ Converted and saved: {len(converted_data)} examples")

## Step 5: Validation

In [ ]:
# Validate dataset
is_valid, errors = validate_dataset(converted_data, required_keys=['text'])
if not is_valid:
    print(f"⚠️ Validation errors found:")
    for error in errors:
        print(f"  - {error}")
else:
    print("✓ Dataset validation passed")

## Step 6: Train/Val Split

Choose one:
- **Option A**: Run split and continue with live data
- **Option B**: Load previously split data

In [ ]:
# OPTION A: Run split (default)
# train_data, val_data = split_dataset(
#     converted_data,
#     train_ratio=config['dataset']['train_ratio'],
#     val_ratio=config['dataset']['val_ratio'],
#     shuffle=True,
#     seed=42,
#     save_dir=project_root / 'data/processed',
#     save_format='json'
# )
# print(f"✓ Split and saved: Train={len(train_data)}, Val={len(val_data)}")

In [5]:
# OPTION B: Load previously split data (uncomment to use)
train_path = project_root / 'data/processed/train.json'
val_path = project_root / 'data/processed/val.json'

if train_path.exists() and val_path.exists():
    with open(train_path, 'r', encoding='utf-8') as f:
        train_data = json.load(f)
    with open(val_path, 'r', encoding='utf-8') as f:
        val_data = json.load(f)
    print(f"✓ Loaded split data: Train={len(train_data)}, Val={len(val_data)}")
else:
    print(f"⚠️ Split data not found, running split...")
    train_data, val_data = split_dataset(
        converted_data,
        train_ratio=config['dataset']['train_ratio'],
        val_ratio=config['dataset']['val_ratio'],
        shuffle=True,
        seed=42,
        save_dir=project_root / 'data/processed',
        save_format='json'
    )
    print(f"✓ Split and saved: Train={len(train_data)}, Val={len(val_data)}")

✓ Loaded split data: Train=402, Val=45


In [6]:
# Convert to HuggingFace Dataset format for training
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data) if val_data else None
print(f"✓ Created HuggingFace datasets for training")

✓ Created HuggingFace datasets for training


## Step 7: Model Loading & Setup

Load the base model and configure LoRA adapters.

In [3]:
# Load model
model, tokenizer = load_model(
    config['model']['name'],
    config,
    max_seq_length=config['training']['max_seq_length']
)
print("✓ Model loaded successfully")

==((====))==  Unsloth 2026.1.4: Fast Lfm2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5070 Ti Laptop GPU. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✓ Model loaded successfully


In [4]:
# Setup LoRA
model = setup_lora(model, config, max_seq_length=config['training']['max_seq_length'])
print("✓ LoRA adapters configured")

Unsloth: Making `model.base_model.model.model` require gradients
✓ LoRA adapters configured


## Step 8: Training

Train the model with your prepared dataset.

In [7]:
# Train model
output_dir = project_root / config['output']['base_dir'] / 'training'
trainer = train_model(
    model,
    tokenizer,
    train_dataset,
    val_dataset,
    config,
    output_dir
)
print(f"✓ Training complete! Checkpoints saved to: {output_dir}")

Unsloth: Tokenizing ["text"]:   0%|          | 0/402 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/45 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 402 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,769,472 of 1,172,110,080 (0.15% trained)


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.591800
2,2.323800
3,2.504500
4,2.319800
5,2.579600
6,2.424600
7,2.518600
8,2.333700
9,2.294400
10,2.320100


✓ Training complete! Checkpoints saved to: d:\LFM-Tuner\outputs\training


## Step 9: Export

Export the trained model in all enabled formats.

In [8]:
# Export in all enabled formats
export_dir = project_root / config['output']['base_dir'] / 'exports'
export_all(
    model,
    tokenizer,
    export_dir,
    config,
    model_name="fine_tuned_model"
)
print(f"\n✓ All exports saved to: {export_dir}")

AttributeError: type object 'FastLanguageModel' has no attribute 'merge_and_unload'

## Summary

Pipeline complete! Your files are organized in:

**Intermediate Data (can resume from any):**
- `data/generated/augmented_dataset.json` - After augmentation
- `data/generated/filtered_dataset.json` - After filtering
- `data/processed/converted_dataset.json` - After format conversion
- `data/processed/train.json` - Training split
- `data/processed/val.json` - Validation split

**Model Outputs:**
- `outputs/training/` - Training checkpoints
- `outputs/exports/` - Exported models in various formats

**Workflow Options:**
1. **Full Pipeline**: Run all cells in order (Option A for each step)
2. **Resume from Checkpoint**: Use Option B to load saved data at any step
3. **Skip Steps**: Comment out steps you don't need
4. **Experiment**: Try different augmentation strategies by loading filtered data and re-running from there